# Session 5, The threshold is a care decision

> **A clinic wants to know which patients to reach out to early**, before they get sick enough to need hospital care. You'll build the model, watch **accuracy lie to you**, then choose the cut-off for “flag for support”, a clinical and ethical decision, not a statistical one.

**How to work:** let Claude write the code; *you* read what the model is doing and make the call.

The data is a representative sample of CDC COVID-19 case records. Each row is one patient.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session5/data/covid_cdc.csv"

import pandas as pd

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload covid_cdc.csv ...")
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    except ImportError:
        df = pd.read_csv("data/covid_cdc.csv")   # local fallback (repo: data/covid_cdc.csv)

df.head()

### About this dataset (read this first)

`covid_cdc.csv` is a **teaching reconstruction** of the US CDC COVID-19 case-surveillance data. It is built to reproduce the real relationships between age, sex, race and hospitalisation in the CDC records, but it is **not** a file of real individual patients, so you'll see many identical rows, and no single row should be read as a real person. The *patterns* are faithful to documented CDC findings (for example, minority patients were hospitalised at younger ages); the individual records are synthetic. Good for learning the method, not a research dataset.

## What the columns mean

One patient per row. The columns:

| Column | Meaning |
|---|---|
| `age_group` | age band, `0 - 9 Years` … `80+ Years` |
| `sex` | Male / Female |
| `race` | patient's race / ethnicity (we set this aside until Notebook 2) |
| `hospitalised` | **the target: did the patient need hospital care? 1 = yes, 0 = no** |

> We predict `hospitalised` from **age and sex only**. Hold that thought, in the fairness notebook it matters that the model never sees `race`.

## Step 1, The baseline: how good is doing nothing?

Before any model: what accuracy do you get by predicting **“no one needs care”** (flag nobody)?

In [ ]:
baseline = (df['hospitalised'] == 0).mean()
print(f'Predict "no one needs care" -> accuracy = {baseline:.1%}')

That's the **accuracy paradox**: a model that flags nobody, helps nobody, and is still ~90% accurate, because ~90% of patients recover at home. Any real model has to beat *that*, and accuracy alone won't tell you whether it does.

## Step 2, Build the model, then look inside

> Ask Claude: *"Fit a logistic regression predicting `hospitalised` from `age_group` and `sex`. One-hot encode them, split 70/30 stratified, and show the accuracy and the confusion matrix."*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y = df['hospitalised']
X = pd.get_dummies(df[['age_group', 'sex']], drop_first=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
pred = clf.predict(Xte)
print('Accuracy:', round(accuracy_score(yte, pred), 3))
ConfusionMatrixDisplay(confusion_matrix(yte, pred),
                       display_labels=['fine', 'needed care']).plot(); plt.show()

The accuracy looks great (~90%). Now read the confusion matrix. At the default 0.5 cut-off the model flags **almost nobody**, the whole right-hand column is nearly empty. High accuracy, and it walks past the patients who actually needed care.

## Step 3, Precision and recall: what accuracy hid

> Ask Claude: *"Print the precision and recall for the patients who needed care."*

- **Precision**, of everyone the model flagged, what share really needed care? (false alarms)
- **Recall**, of everyone who needed care, what share did we flag? (missed patients)

In [ ]:
from sklearn.metrics import precision_score, recall_score
print('Precision:', round(precision_score(yte, pred, zero_division=0), 3))
print('Recall:   ', round(recall_score(yte, pred), 3))

Recall is very low at the default cut-off. The model is 90% accurate and misses almost every patient who needed care, because accuracy is dominated by the 90% who were always going to be fine.

## Step 4, Move the threshold  🎯

The model doesn't really output yes/no, it outputs a **risk score** (a probability). `predict` just cuts it at 0.5. **You** choose the cut-off. Predict first: *to catch more at-risk patients, do you move it up or down?*

> Ask Claude: *"Using predicted probabilities, show precision, recall and the share of patients flagged at cut-offs 0.20, 0.15 and 0.10."*

In [ ]:
prob = clf.predict_proba(Xte)[:, 1]
print(f"{'cut-off':>8} | {'precision':>9} | {'recall':>7} | {'% flagged':>9}")
for t in [0.20, 0.15, 0.10]:
    p = (prob >= t).astype(int)
    prec = precision_score(yte, p, zero_division=0)
    rec = recall_score(yte, p)
    print(f'{t:>8} | {prec:>9.3f} | {rec:>7.3f} | {p.mean():>9.1%}')

Lower the cut-off and **recall rises** (you catch more at-risk patients) but **precision falls** (more false alarms) and you **flag more people** for outreach. None of that is a statistics question.

## Step 5, Decide (this is the point)

Two forces pin down the cut-off. **Capacity:** the clinic can only follow up so many patients. **Cost:** a missed patient who needed care is far worse than a false alarm. Start with capacity, it's the one you can't argue your way out of.

### 5a, You can follow up 15% of patients. Which 15%?

Flag the 15% highest-risk patients, and see how many of the at-risk patients that actually catches.

In [ ]:
import numpy as np
capacity = 0.15                          # you can reach 15% of patients this week
cut = np.quantile(prob, 1 - capacity)    # the risk score that flags the top 15%
p = (prob >= cut).astype(int)
caught = int(((p == 1) & (yte == 1)).sum()); need = int((yte == 1).sum())
print(f'Flag top {capacity:.0%}  ->  cut-off {cut:.3f}, catch {caught}/{need} at-risk = {caught/need:.0%}')

Change `capacity` to 0.25 or 0.10 and re-run. More capacity catches more at-risk patients, but you reach further down the risk list, so a bigger share of those calls are false alarms. That trade-off is the whole game.

### 5b, Or price the two mistakes and let the cost decide

A missed patient costs more than a false alarm. Try a few price ratios and find the least-harm cut-off.

In [ ]:
for cost_miss, cost_alarm in [(100, 1), (50, 10), (20, 10)]:
    best_t, best_cost = None, None
    for t in np.linspace(0.02, 0.6, 60):
        p = (prob >= t).astype(int)
        fn = int(((p == 0) & (yte == 1)).sum())     # needed care, not flagged
        fp = int(((p == 1) & (yte == 0)).sum())     # flagged, was fine
        cost = fn * cost_miss + fp * cost_alarm
        if best_cost is None or cost < best_cost:
            best_t, best_cost = t, cost
    flagged = (prob >= best_t).mean()
    print(f'miss={cost_miss:>3}, alarm={cost_alarm:>2}  ->  least-harm cut-off {best_t:.2f}, flags {flagged:.0%}')

> ⚠️ **Look at the first row.** When a miss costs 100 and a false alarm costs 1 *with no capacity limit*, the least-harm cut-off collapses to almost zero, the maths says **flag everyone**. That's not a bug, it's the honest answer when misses dominate and outreach is free. But outreach is **never** free, which is exactly why the capacity limit in 5a matters. Real systems use both a cost view and a capacity view.

**Write your recommendation to the clinical lead in three sentences:** the cut-off (or the capacity) you'd run, why, and one caveat. Then change the capacity or the costs and watch your answer move. *That* is why the threshold is a human decision, not a setting.

In [ ]:
# Your three-sentence recommendation to the clinical lead:
#

## Before you leave

1. Why is accuracy the wrong headline number for this problem?
2. In one sentence: what does moving the threshold down buy you, and what does it cost?
3. This model uses only age and sex. Name one thing you'd want to add before trusting it with a real patient, and one risk that comes with adding it.

## Take it to your project

You did this on COVID data because the two outcomes are clean. The same two skills, **choosing a cut-off** and **auditing for fairness**, carry straight to your Brexit group project. When you build the logistic “did this area vote Leave” model (the inside-the-UK method), you pick a cut-off the same way, and you can audit whether it predicts equally well across regions. Want a second rep first? The **bank marketing dataset** we use in class is the natural practice ground: find the “do nothing” baseline, build the model, move the threshold, and pick a cut-off for a stated cost (a subscription is worth far more than a wasted call). The skill transfers exactly.